# 🩺 Liver Disease Prediction System — Maximum Enhancement
**Authors:** Aditya Chhikara  
**Dataset:** Indian Liver Patient Records (ILPD)  
**Stack:** Sklearn · XGBoost · LightGBM · Optuna · SHAP · Seaborn · Plotly  

### What's new vs. the original
| Feature | Original | This notebook |
|---|---|---|
| Models | LR, KNN, SVM, RF | + XGBoost, LightGBM, GBM, Extra Trees |
| Tuning | None | Optuna Bayesian HPO (100 trials) |
| Evaluation | Accuracy only | Acc · AUC-ROC · F1 · MCC · Brier score |
| Explainability | None | SHAP waterfall + beeswarm + force plots |
| Cross-validation | None | Stratified 5-fold per model |
| Imbalance | None | SMOTE oversampling |
| Calibration | None | Platt scaling / isotonic regression |
| Visualisation | Basic matplotlib | Plotly interactive + enhanced seaborn |


In [ ]:
# ── Install all dependencies ──────────────────────────────────────────────
!pip install -q xgboost lightgbm optuna shap imbalanced-learn plotly kagglehub


In [ ]:
import warnings, os, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score, learning_curve)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier, VotingClassifier, StackingClassifier)
from sklearn.naive_bayes import GaussianNB

import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    matthews_corrcoef, brier_score_loss, log_loss)

from imblearn.over_sampling import SMOTE
import shap

np.random.seed(42)
print("✅ All libraries imported successfully")


In [ ]:
# ── Download dataset via kagglehub ───────────────────────────────────────
import kagglehub
path = kagglehub.dataset_download("uciml/indian-liver-patient-records")
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print("Files:", csv_files)
CSV_PATH = os.path.join(path, csv_files[0])


In [ ]:
# ── STEP 1 · Load & Inspect Dataset ──────────────────────────────────────
df = pd.read_csv(CSV_PATH)

# Standardise column names (handles different CSV versions)
df.columns = [c.strip().replace(' ', '_') for c in df.columns]

# Rename target column to 'Target' for consistency
TARGET = [c for c in df.columns if c.lower() in ('dataset','result','target','diagnosis','class')
          or 'disease' in c.lower()]
TARGET = TARGET[0] if TARGET else df.columns[-1]
df.rename(columns={TARGET: 'Target'}, inplace=True)
TARGET = 'Target'

# Binarise: 1 = liver disease, 0 = healthy
if df[TARGET].nunique() == 2 and df[TARGET].max() == 2:
    df[TARGET] = df[TARGET].map({1: 1, 2: 0})

print(f"Shape: {df.shape}")
print(f"Target column: {TARGET}")
print(f"Class distribution:\n{df[TARGET].value_counts()}")
display(df.head())
display(df.describe().round(3))


In [ ]:
# ── STEP 2 · Exploratory Data Analysis ───────────────────────────────────
print("Missing values per column:")
display(df.isnull().sum().to_frame('Missing'))

# ── Class imbalance ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
vc = df[TARGET].value_counts()
axes[0].pie(vc, labels=['Disease','Healthy'], autopct='%1.1f%%',
            colors=['#E74C3C','#2ECC71'], startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Class Balance', fontsize=13, fontweight='bold')

vc.plot(kind='bar', ax=axes[1], color=['#E74C3C','#2ECC71'], edgecolor='black')
axes[1].set_title('Class Counts', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(['Disease','Healthy'], rotation=0)
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── Numeric feature distributions ────────────────────────────────────────
numeric_cols = df.select_dtypes(include=np.number).columns.difference([TARGET]).tolist()
n = len(numeric_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
axes = axes.flatten()
palette = {1: '#E74C3C', 0: '#2ECC71'}

for i, col in enumerate(numeric_cols):
    for cls, color in palette.items():
        subset = df[df[TARGET] == cls][col].dropna()
        axes[i].hist(subset, bins=30, alpha=0.6, color=color,
                     label=f'{"Disease" if cls==1 else "Healthy"}', edgecolor='none')
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].grid(alpha=0.3)

for j in range(n, len(axes)):
    axes[j].axis('off')

plt.suptitle('Feature Distributions by Class', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── Correlation heatmap ──────────────────────────────────────────────────
plt.figure(figsize=(12, 9))
corr = df[numeric_cols + [TARGET]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, square=True, linewidths=0.5,
            annot_kws={'size': 8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Plotly interactive version
fig_corr = px.imshow(corr, text_auto='.2f', aspect='auto',
                     color_continuous_scale='RdYlGn',
                     title='Interactive Correlation Heatmap')
fig_corr.show()


In [ ]:
# ── STEP 3 · Preprocessing ───────────────────────────────────────────────
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].values.astype(int)

# Encode gender (or any categorical column)
cat_cols = X.select_dtypes(include='object').columns.tolist()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# Impute missing values (median strategy)
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

FEATURE_NAMES = X.columns.tolist()
print(f"Features ({len(FEATURE_NAMES)}):", FEATURE_NAMES)

# Train / test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── SMOTE to handle class imbalance ──────────────────────────────────────────
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_sc, y_train)
print(f"After SMOTE — train size: {X_train_res.shape[0]}")
print(f"Class balance: {np.bincount(y_train_res)}")


In [ ]:
# ── STEP 4 · Define & Cross-Validate Models ──────────────────────────────
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

base_models = {
    "Logistic Regression":  LogisticRegression(max_iter=2000, C=1.0),
    "KNN":                  KNeighborsClassifier(n_neighbors=7),
    "SVM (RBF)":            SVC(kernel='rbf', probability=True, C=1.0),
    "Naive Bayes":          GaussianNB(),
    "Random Forest":        RandomForestClassifier(n_estimators=200, random_state=42),
    "Extra Trees":          ExtraTreesClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting":    GradientBoostingClassifier(n_estimators=200, random_state=42),
    "XGBoost":              xgb.XGBClassifier(n_estimators=200, eval_metric='logloss',
                                               use_label_encoder=False, random_state=42),
    "LightGBM":             lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1),
}

cv_results = {}
print(f"{'Model':<25} {'AUC-ROC':>8}  {'F1':>7}  {'Accuracy':>9}")
print("-" * 55)

for name, model in base_models.items():
    auc_scores  = cross_val_score(model, X_train_res, y_train_res, cv=SKF,
                                  scoring='roc_auc', n_jobs=-1)
    f1_scores   = cross_val_score(model, X_train_res, y_train_res, cv=SKF,
                                  scoring='f1', n_jobs=-1)
    acc_scores  = cross_val_score(model, X_train_res, y_train_res, cv=SKF,
                                  scoring='accuracy', n_jobs=-1)
    cv_results[name] = {
        'auc_mean': auc_scores.mean(), 'auc_std': auc_scores.std(),
        'f1_mean':  f1_scores.mean(),
        'acc_mean': acc_scores.mean(),
    }
    print(f"{name:<25} {auc_scores.mean():.4f}±{auc_scores.std():.3f}  "
          f"{f1_scores.mean():.4f}  {acc_scores.mean():.4f}")

print("\n✅ Cross-validation complete")


In [ ]:
# ── STEP 5 · Bayesian HPO with Optuna (XGBoost & LightGBM) ──────────────

def objective_xgb(trial):
    params = {
        'n_estimators':   trial.suggest_int('n_estimators', 100, 500),
        'max_depth':      trial.suggest_int('max_depth', 3, 9),
        'learning_rate':  trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample':      trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':      trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':     trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'eval_metric': 'logloss', 'use_label_encoder': False, 'random_state': 42,
    }
    m = xgb.XGBClassifier(**params)
    return cross_val_score(m, X_train_res, y_train_res, cv=3,
                           scoring='roc_auc', n_jobs=-1).mean()

def objective_lgb(trial):
    params = {
        'n_estimators':   trial.suggest_int('n_estimators', 100, 500),
        'max_depth':      trial.suggest_int('max_depth', 3, 9),
        'learning_rate':  trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'num_leaves':     trial.suggest_int('num_leaves', 20, 150),
        'subsample':      trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':      trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'verbose': -1, 'random_state': 42,
    }
    m = lgb.LGBMClassifier(**params)
    return cross_val_score(m, X_train_res, y_train_res, cv=3,
                           scoring='roc_auc', n_jobs=-1).mean()

print("Tuning XGBoost (100 trials)…")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=100, show_progress_bar=False)
best_xgb_params = study_xgb.best_params
best_xgb_params.update({'eval_metric':'logloss','use_label_encoder':False,'random_state':42})
print(f"  Best XGB AUC: {study_xgb.best_value:.4f}")

print("Tuning LightGBM (100 trials)…")
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=100, show_progress_bar=False)
best_lgb_params = study_lgb.best_params
best_lgb_params.update({'verbose':-1,'random_state':42})
print(f"  Best LGB AUC: {study_lgb.best_value:.4f}")


In [ ]:
# ── STEP 6 · Train Tuned Models + Stacking Ensemble ─────────────────────

tuned_xgb = xgb.XGBClassifier(**best_xgb_params)
tuned_lgb = lgb.LGBMClassifier(**best_lgb_params)
tuned_rf  = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)
tuned_et  = ExtraTreesClassifier(n_estimators=300, max_depth=8, random_state=42)

# Stacking ensemble: XGB + LGB + RF + ET → Logistic Regression meta-learner
stacking = StackingClassifier(
    estimators=[
        ('xgb', tuned_xgb),
        ('lgb', tuned_lgb),
        ('rf',  tuned_rf),
        ('et',  tuned_et),
    ],
    final_estimator=LogisticRegression(max_iter=2000),
    cv=5, passthrough=False, n_jobs=-1
)

# Voting ensemble (soft)
voting = VotingClassifier(
    estimators=[('xgb', tuned_xgb), ('lgb', tuned_lgb),
                ('rf',  tuned_rf),  ('et',  tuned_et)],
    voting='soft', n_jobs=-1
)

all_models = {
    **{k: v for k,v in base_models.items()},
    "Tuned XGBoost":  tuned_xgb,
    "Tuned LightGBM": tuned_lgb,
    "Stacking":       stacking,
    "Soft Voting":    voting,
}

print("Training all models on SMOTE data…")
for name, m in all_models.items():
    m.fit(X_train_res, y_train_res)
    print(f"  ✔ {name}")
print("\n✅ All models trained")


In [ ]:
# ── STEP 7 · Full Evaluation on Held-Out Test Set ────────────────────────

results_full = {}
for name, m in all_models.items():
    y_pred  = m.predict(X_test_sc)
    y_proba = m.predict_proba(X_test_sc)[:, 1] if hasattr(m, 'predict_proba') else None

    results_full[name] = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred, zero_division=0),
        'f1':        f1_score(y_test, y_pred, zero_division=0),
        'mcc':       matthews_corrcoef(y_test, y_pred),
        'auc_roc':   roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
        'avg_prec':  average_precision_score(y_test, y_proba) if y_proba is not None else np.nan,
        'brier':     brier_score_loss(y_test, y_proba) if y_proba is not None else np.nan,
        'log_loss':  log_loss(y_test, y_proba) if y_proba is not None else np.nan,
    }

results_df = pd.DataFrame(results_full).T.round(4)
results_df = results_df.sort_values('auc_roc', ascending=False)
print("\n📊 Full Evaluation Results (sorted by AUC-ROC):")
display(results_df.style.background_gradient(cmap='YlGn', subset=['accuracy','f1','auc_roc','mcc'])
                         .background_gradient(cmap='YlOrRd_r', subset=['brier','log_loss']))

BEST_MODEL_NAME = results_df['auc_roc'].idxmax()
BEST_MODEL = all_models[BEST_MODEL_NAME]
print(f"\n🏆 Best Model: {BEST_MODEL_NAME}  (AUC-ROC = {results_df.loc[BEST_MODEL_NAME,'auc_roc']:.4f})")


In [ ]:
# ── ROC Curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cmap = plt.cm.get_cmap('tab20', len(all_models))

for i, (name, m) in enumerate(all_models.items()):
    if not hasattr(m, 'predict_proba'): continue
    y_pr = m.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pr)
    roc_a = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=cmap(i), lw=1.5, label=f"{name} ({roc_a:.3f})")

    prec, rec, _ = precision_recall_curve(y_test, y_pr)
    ap = average_precision_score(y_test, y_pr)
    axes[1].plot(rec, prec, color=cmap(i), lw=1.5, label=f"{name} (AP={ap:.3f})")

for ax, title, xl, yl in [
    (axes[0], 'ROC Curves', 'False Positive Rate', 'True Positive Rate'),
    (axes[1], 'Precision-Recall Curves', 'Recall', 'Precision')]:
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.grid(alpha=0.3); ax.legend(fontsize=7, loc='lower right')

axes[0].plot([0,1],[0,1],'k--', lw=1, label='Random')
plt.tight_layout(); plt.show()


In [ ]:
# ── Confusion Matrices (top 4 by AUC-ROC) ────────────────────────────────
top4 = results_df.head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, name in enumerate(top4):
    m = all_models[name]
    y_pred = m.predict(X_test_sc)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp+fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn+fp) > 0 else 0

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Healthy','Disease'],
                yticklabels=['Healthy','Disease'],
                annot_kws={'size':14})
    axes[i].set_title(
        f"{name}\nAUC={results_df.loc[name,'auc_roc']:.3f}  "
        f"Sen={sensitivity:.3f}  Spe={specificity:.3f}",
        fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Actual'); axes[i].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices – Top 4 Models', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── Interactive Model Comparison (Plotly) ────────────────────────────────
metrics = ['accuracy', 'f1', 'auc_roc', 'mcc']
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[m.upper() for m in metrics])

for idx, metric in enumerate(metrics):
    row, col = divmod(idx, 2)
    sorted_r = results_df.sort_values(metric, ascending=False)
    colors = ['#E74C3C' if n == BEST_MODEL_NAME else '#3498DB'
              for n in sorted_r.index]
    fig.add_trace(go.Bar(
        x=sorted_r.index, y=sorted_r[metric],
        marker_color=colors, name=metric, showlegend=False,
        text=sorted_r[metric].round(3), textposition='outside'
    ), row=row+1, col=col+1)

fig.update_layout(height=700, title_text='Model Performance Comparison',
                  title_font_size=16, template='plotly_white')
fig.show()


In [ ]:
# ── Learning Curves (Best Model) ─────────────────────────────────────────
train_sizes, train_scores, val_scores = learning_curve(
    BEST_MODEL, X_train_res, y_train_res,
    cv=5, scoring='roc_auc', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10))

t_mean, t_std = train_scores.mean(1), train_scores.std(1)
v_mean, v_std = val_scores.mean(1),   val_scores.std(1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, t_mean, 'o-', color='#E74C3C', label='Train AUC')
plt.fill_between(train_sizes, t_mean-t_std, t_mean+t_std, alpha=0.15, color='#E74C3C')
plt.plot(train_sizes, v_mean, 'o-', color='#2ECC71', label='CV AUC')
plt.fill_between(train_sizes, v_mean-v_std, v_mean+v_std, alpha=0.15, color='#2ECC71')
plt.title(f'Learning Curves — {BEST_MODEL_NAME}', fontsize=14, fontweight='bold')
plt.xlabel('Training Samples'); plt.ylabel('AUC-ROC')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# ── STEP 8 · SHAP Explainability ─────────────────────────────────────────
# Use XGBoost / LightGBM if available (TreeExplainer is fastest)
shap_model_name = "Tuned XGBoost" if "Tuned XGBoost" in all_models else BEST_MODEL_NAME
shap_model = all_models[shap_model_name]

try:
    explainer = shap.TreeExplainer(shap_model)
    shap_values = explainer(pd.DataFrame(X_test_sc, columns=FEATURE_NAMES))
except Exception:
    # Fall back to KernelExplainer for non-tree models
    bg = shap.maskers.Independent(
        pd.DataFrame(X_train_res[:100], columns=FEATURE_NAMES))
    explainer = shap.Explainer(shap_model.predict_proba, bg)
    shap_values = explainer(pd.DataFrame(X_test_sc, columns=FEATURE_NAMES))

# ── Beeswarm ─────────────────────────────────────────────────────────────────
plt.figure()
shap.plots.beeswarm(shap_values if len(shap_values.shape)==2
                    else shap_values[:,:,1],
                    show=True, max_display=15)
plt.suptitle(f'SHAP Beeswarm — {shap_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── SHAP Bar (mean |SHAP|) ───────────────────────────────────────────────
plt.figure()
shap.plots.bar(shap_values if len(shap_values.shape)==2
               else shap_values[:,:,1],
               max_display=15, show=True)
plt.suptitle(f'SHAP Feature Importance — {shap_model_name}',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── SHAP Waterfall for a single patient ──────────────────────────────────
idx = 0   # change to inspect any test sample
sv = shap_values if len(shap_values.shape)==2 else shap_values[:,:,1]
plt.figure()
shap.plots.waterfall(sv[idx], show=True)
plt.title(f'Patient #{idx} — Prediction Explanation', fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── STEP 9 · Probability Calibration ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot([0,1], [0,1], 'k--', label='Perfectly calibrated')

for name in [BEST_MODEL_NAME, "Tuned XGBoost", "Tuned LightGBM", "Stacking",
             "Logistic Regression"]:
    if name not in all_models: continue
    m = all_models[name]
    if not hasattr(m, 'predict_proba'): continue
    y_pr = m.predict_proba(X_test_sc)[:, 1]
    fraction_pos, mean_pred_val = calibration_curve(y_test, y_pr, n_bins=10)
    ax.plot(mean_pred_val, fraction_pos, 'o-', lw=2, label=name)

ax.set_xlabel('Mean Predicted Probability'); ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Plot', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Calibrate best model (isotonic regression)
cal_model = CalibratedClassifierCV(BEST_MODEL, cv='prefit', method='isotonic')
cal_model.fit(X_test_sc, y_test)
print("Best model calibrated ✅")


In [ ]:
# ── STEP 10 · Predict New Patient ────────────────────────────────────────
sample_patient = {
    "Age": 45,
    "Gender": "Male",
    "Total_Bilirubin": 0.7,
    "Direct_Bilirubin": 0.1,
    "Alkaline_Phosphotase": 187,
    "Alamine_Aminotransferase": 16,
    "Aspartate_Aminotransferase": 18,
    "Total_Protiens": 6.8,
    "Albumin": 3.3,
    "Albumin_and_Globulin_Ratio": 0.9
}

def predict_new_patient(patient_dict, model, scaler, imputer, label_encoders,
                        feature_names, cal_model=None):
    pat_df = pd.DataFrame([patient_dict])
    
    # Encode categoricals
    for col, le in label_encoders.items():
        if col in pat_df.columns:
            val = pat_df[col].astype(str).values[0]
            if val in le.classes_:
                pat_df[col] = le.transform([val])
            else:
                pat_df[col] = 0  # unknown category → 0

    # Ensure same columns
    for col in feature_names:
        if col not in pat_df.columns:
            pat_df[col] = 0.0
    pat_df = pat_df[feature_names]
    
    # Impute → scale
    pat_imp = pd.DataFrame(imputer.transform(pat_df), columns=feature_names)
    pat_sc  = scaler.transform(pat_imp)
    
    # Predict
    clf = cal_model if cal_model else model
    pred      = clf.predict(pat_sc)[0]
    proba     = clf.predict_proba(pat_sc)[0]
    
    return int(pred), float(proba[1])

pred, prob = predict_new_patient(
    sample_patient, BEST_MODEL, scaler, imputer, label_encoders,
    FEATURE_NAMES, cal_model=cal_model)

print(f"\n{'='*50}")
print(f"🩺  Patient Liver Disease Risk Assessment")
print(f"{'='*50}")
print(f"Model used   : {BEST_MODEL_NAME}")
print(f"Prediction   : {'⚠️  LIVER DISEASE' if pred==1 else '✅  HEALTHY'}")
print(f"Disease prob : {prob:.1%}")
print(f"Healthy prob : {1-prob:.1%}")
print(f"{'='*50}")


In [ ]:
# ── STEP 11 · Final Summary ──────────────────────────────────────────────
print("\n" + "="*70)
print("                  FINAL PERFORMANCE SUMMARY")
print("="*70)
display(results_df[['accuracy','precision','recall','f1','mcc','auc_roc','brier']]
        .style.background_gradient(cmap='YlGn', subset=['accuracy','f1','auc_roc','mcc'])
               .background_gradient(cmap='YlOrRd_r', subset=['brier'])
               .format("{:.4f}"))
print(f"\n🏆 Best model: {BEST_MODEL_NAME}")
print(f"   AUC-ROC  = {results_df.loc[BEST_MODEL_NAME,'auc_roc']:.4f}")
print(f"   F1-Score = {results_df.loc[BEST_MODEL_NAME,'f1']:.4f}")
print(f"   Accuracy = {results_df.loc[BEST_MODEL_NAME,'accuracy']:.4f}")
print(f"   MCC      = {results_df.loc[BEST_MODEL_NAME,'mcc']:.4f}")
